In [0]:
%sql
-- ============================================================================
-- Pipeline Step: Gold Dimension - dim_project
-- Description: Projects dimension sourced from Silver Reference data (by Jan).
-- Dependencies: dbr_dev.wikimediademo_bronze.ref_wikimedia_projects (or Silver)
-- ============================================================================

USE CATALOG dbr_dev;
USE SCHEMA wikimediademo_gold;

CREATE OR REPLACE TABLE dbr_dev.wikimediademo_gold.dim_project AS
SELECT 
    -- Surrogate primary key
    md5(coalesce(wiki_code, 'unknown')) AS project_key,
    wiki_code,
    coalesce(language_code, 'unknown') AS language_code,
    coalesce(project_name, 'Wikipedia') AS project_name,
    site_url,
    current_timestamp() AS gold_updated_at
FROM dbr_dev.wikimediademo_bronze.ref_wikimedia_projects ---------- when Jan will make SCD Type 2 pipline we can change it
-- Filter for current records if SCD Type 2 is enabled in Silver
-- WHERE is_current = true OR is_current IS NULL
;

-- Insert an unknown/default project record for orphan prevention
INSERT INTO dbr_dev.wikimediademo_gold.dim_project (project_key, wiki_code, language_code, project_name, site_url, gold_updated_at)
VALUES (md5('unknown'), 'unknown', 'unknown', 'Other Wikimedia Project', 'https://www.wikimedia.org', current_timestamp());


In [0]:
%sql

-- Verification
SELECT * FROM dbr_dev.wikimediademo_gold.dim_project LIMIT 5;

In [0]:
%sql
SELECT count(*) FROM dbr_dev.wikimediademo_gold.dim_project;